In [ ]:
"""
Datasets are available at https://zenodo.org/records/6355684 
"""

'\nDatasets are available at https://zenodo.org/records/6355684 \n'

In [2]:
import sys
print(sys.version)

3.10.11 (v3.10.11:7d4cc5aa85, Apr  4 2023, 19:05:19) [Clang 13.0.0 (clang-1300.0.29.30)]


In [3]:
import numpy as np
from sklearn.metrics import silhouette_score
import kmedoids
from scipy.spatial.distance import squareform, pdist
import time
from common import get_df_sampled, eval_ub_runtime
from pathlib import Path

In [4]:
base_path = Path.cwd()

In [5]:
# generate results folder
results_dir = base_path / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# 1. ALOI HSB 5x5x5

In [6]:
dataset_name = "aloi-hsb-5x5x5"
N_FEATURES = 125

In [7]:
df_sampled = get_df_sampled(dataset_name=dataset_name, class_size=40)
df_sampled.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,class
0,0.045401,0.001526,0.003160,0.006736,0.023887,0.172085,0.004343,0.011235,0.008873,0.003832,...,0.000411,0.000066,0.000335,0.002301,0.009011,0.006079,0.000045,"""img1""","""1/1_r280.png""",1
1,0.045225,0.000613,0.001031,0.002921,0.020752,0.173089,0.002421,0.004028,0.005229,0.005391,...,0.000271,0.000416,0.000201,0.000520,0.004795,0.011194,0.002145,"""img1""","""1/1_i230.png""",1
2,0.014931,0.000016,0.000000,0.000000,0.000301,0.130830,0.000723,0.000744,0.001067,0.020598,...,0.000120,0.000373,0.000179,0.000626,0.003092,0.012356,0.005875,"""img1""","""1/1_i150.png""",1
3,0.043287,0.001560,0.003798,0.005082,0.019194,0.170754,0.006427,0.012146,0.007935,0.003599,...,0.000330,0.000183,0.000183,0.001350,0.007388,0.009465,0.000136,"""img1""","""1/1_r305.png""",1
4,0.050345,0.001476,0.003187,0.005841,0.028332,0.177456,0.003007,0.005674,0.006933,0.004853,...,0.000543,0.000079,0.000104,0.002783,0.017675,0.003879,0.000070,"""img1""","""1/1_r210.png""",1


In [8]:
y = df_sampled["class"]
X = df_sampled.iloc[:, 0:N_FEATURES]
X = X.to_numpy(dtype='float32')
print(X.shape, X.dtype)

(40000, 125) float32


In [9]:
D = squareform(pdist(X, metric="cityblock")).astype(np.float32)

In [10]:
print(D.shape, D.dtype)

(40000, 40000) float32


### 1.1 fastmsc for K = 1000

In [11]:
start = time.perf_counter()
cluster_labels = (kmedoids.fastmsc(diss=D, medoids=1000, random_state=42).labels + 1) 
print(f"RT = {(time.perf_counter() - start) / 1000}")

RT = 1.0833736399998888


In [12]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
1000.000  2.000     0.267     0.557     0.133     0.914     0.854     0.231     0.913     0.854     0.249     


### 1.2 fastmsc for $K=100, 200, ..., 800, 900$

In [13]:
try:
    cluster_labels_list = np.load(f"{results_dir}/aloi125-fastmsc-dagger-cluster-labels-list.npy")
except:
    start = time.perf_counter()
    cluster_labels_list = [(kmedoids.fastmsc(diss=D, medoids=n_medoids, random_state=42).labels + 1) for n_medoids in [100, 200, 300, 400, 500, 600, 700, 800, 900]]
    print(f"RT = {(time.perf_counter() - start) / 1000}")
    # Save cluster labels
    np.save(f"{results_dir}/aloi125-fastmsc-dagger-cluster-labels-list.npy", cluster_labels_list)

In [14]:
asw_list = [silhouette_score(X=D, labels=cluster_labels, metric='precomputed') for cluster_labels in cluster_labels_list]
asw_list

[np.float32(0.18102254),
 np.float32(0.12757215),
 np.float32(0.10489464),
 np.float32(0.09952153),
 np.float32(0.10900171),
 np.float32(0.114229664),
 np.float32(0.1204823),
 np.float32(0.12539692),
 np.float32(0.12986174)]

In [15]:
argm = np.argmax(asw_list)
argm 

np.int64(0)

In [16]:
cluster_labels = cluster_labels_list[argm]

In [17]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
100.000   7.000     0.006     0.368     0.181     0.914     0.802     0.208     0.866     0.791     0.236     


### 1.3 dynmsc for K in range 2 to 50

In [18]:
start = time.perf_counter()
cluster_labels = (kmedoids.dynmsc(diss=D, medoids=50, random_state=42).labels + 1) 
print(f"RT = {(time.perf_counter() - start) / 1000}")

RT = 1.6965973837079946


In [19]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
2.000     976.000   0.000     0.009     0.502     0.914     0.451     0.227     0.618     0.188     0.242     


# 2. ALOI HSB 14x6x6

In [20]:
dataset_name = "aloi-hsb-14x6x6"
N_FEATURES = 504

In [21]:
df_sampled = get_df_sampled(dataset_name=dataset_name, class_size=40)
df_sampled.head()

,0,1,2,3,4,5,6,7,8,9,...,497,498,499,500,501,502,503,504,505,class
0,0.010166,0.000084,0.000045,0.000079,0.000129,0.001203,0.034017,0.000197,0.000805,0.001309,...,0.000014,0.000061,0.001341,0.003411,0.009531,0.002423,0.000007,"""img1""","""1/1_r280.png""",1
1,0.010283,0.000054,0.000068,0.000059,0.000061,0.001610,0.032213,0.000167,0.000158,0.000174,...,0.000411,0.000050,0.000251,0.001487,0.007643,0.007541,0.000925,"""img1""","""1/1_i230.png""",1
2,0.004230,0.000000,0.000000,0.000000,0.000000,0.000000,0.029790,0.000052,0.000036,0.000009,...,0.000434,0.000043,0.000488,0.001069,0.005098,0.010948,0.003913,"""img1""","""1/1_i150.png""",1
3,0.010306,0.000093,0.000070,0.000079,0.000090,0.001072,0.032052,0.000278,0.000963,0.000726,...,0.000059,0.000086,0.000554,0.002062,0.009861,0.005012,0.000016,"""img1""","""1/1_r305.png""",1
4,0.011635,0.000131,0.000149,0.000156,0.000104,0.001216,0.036904,0.000416,0.000450,0.000877,...,0.000043,0.000084,0.001225,0.005269,0.016473,0.000328,0.000045,"""img1""","""1/1_r210.png""",1


In [22]:
y = df_sampled["class"]
X = df_sampled.iloc[:, 0:N_FEATURES]
X = X.to_numpy(dtype='float32')
print(X.shape, X.dtype)

(40000, 504) float32


In [23]:
D = squareform(pdist(X, metric="cityblock")).astype(np.float32)

In [24]:
print(D.shape, D.dtype)

(40000, 40000) float32


### 2.1 fastmsc for K = 1000

In [25]:
start = time.perf_counter()
cluster_labels = (kmedoids.fastmsc(diss=D, medoids=1000, random_state=42).labels + 1) 
print(f"RT = {(time.perf_counter() - start) / 1000}")

RT = 1.0866431474168785


In [26]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
1000.000  2.000     0.292     0.589     0.142     0.901     0.842     0.231     0.900     0.842     0.242     


### 2.2 fastmsc for $K=100, 200, ..., 800, 900$

In [27]:
try:
    cluster_labels_list = np.load(f"{results_dir}/aloi504-fastmsc-dagger-cluster-labels-list.npy")
except:
    start = time.perf_counter()
    cluster_labels_list = [(kmedoids.fastmsc(diss=D, medoids=n_medoids, random_state=42).labels + 1) for n_medoids in [100, 200, 300, 400, 500, 600, 700, 800, 900]]
    print(f"RT = {(time.perf_counter() - start) / 1000}")
    # Save cluster labels
    np.save(f"{results_dir}/aloi504-fastmsc-dagger-cluster-labels-list.npy", cluster_labels_list)

In [28]:
asw_list = [silhouette_score(X=D, labels=cluster_labels, metric='precomputed') for cluster_labels in cluster_labels_list]
asw_list

[np.float32(0.16062114),
 np.float32(0.117390014),
 np.float32(0.10027606),
 np.float32(0.09333442),
 np.float32(0.09996997),
 np.float32(0.110528514),
 np.float32(0.120030716),
 np.float32(0.12794694),
 np.float32(0.13597922)]

In [29]:
argm = np.argmax(asw_list)
argm 

np.int64(0)

In [30]:
cluster_labels = cluster_labels_list[argm]

In [31]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
100.000   3.000     0.005     0.352     0.161     0.901     0.822     0.215     0.883     0.818     0.240     


### 2.3 dynmsc for K in range 2 to 50

In [32]:
start = time.perf_counter()
cluster_labels = (kmedoids.dynmsc(diss=D, medoids=50, random_state=42).labels + 1) 
print(f"RT = {(time.perf_counter() - start) / 1000}")

RT = 1.8259239101670683


In [33]:
eval_ub_runtime(D, y, cluster_labels)

Clusters  Min|C_I|  ARI       AMI       ASW       UB        WCRE      RT        UB_C      UB_C_WCRE UB_C_RT   
--------------------------------------------------------------------------------------------------------------
2.000     38.000    0.000     0.001     0.470     0.901     0.478     0.255     0.752     0.375     0.282     
